# Stage 2 假陰性人工核對

對照 STAGE2_PLAN.md 第 1.2 / 6 節：某幾類 val AUROC 偏低，要先判斷是「模型真的沒學好」還是
「切片選取邏輯（病灶面積最大的 z 切面）本來就看不出這個標籤」，不要急著怪模型。

用法：
1. 跑第一個 code cell 載入資料
2. 在第二個 cell 的下拉選單選一個要核對的 label（只列出 `eval_false_negatives.csv` 裡有 dump 的類別）
3. 跑第三個 cell 開始看圖，對每一張假陰性切片按對應的按鈕：
   - **看得到病灶（模型問題）**：肉眼/報告描述都能定位到病灶，模型卻預測低機率 → 模型沒學好
   - **看不到病灶（切片天花板）**：這張切面本來就看不出這個病灶（例如需要別的軸切面才看得到）→ 資料選取限制，不是模型的錯
   - **不確定**：需要更專業判讀或看其他切面才能確定
4. 判斷結果會即時存到 `checkpoints/manual_review_results.csv`，中斷後重跑也會跳過已經判斷過的
5. 換 label 時，重新跑第二格（換下拉選單的值）再跑第三格即可

In [ ]:
import csv
import json
import os
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

ROOT_DIR = Path("/root/Desktop/VLM")
REX_DIR = ROOT_DIR / "data" / "processed_rex"
CHECKPOINT_DIR = ROOT_DIR / "U-VLM" / "stage2" / "checkpoints"
FALSE_NEG_CSV = CHECKPOINT_DIR / "eval_false_negatives.csv"
RESULTS_CSV = CHECKPOINT_DIR / "manual_review_results.csv"
REVIEWER = "tony.tu@daikso.com"


def load_manifest():
    manifest = {}
    with open(REX_DIR / "manifest.jsonl") as f:
        for line in f:
            r = json.loads(line)
            manifest[r["volume_id"]] = r
    return manifest


def full_report_text(record):
    rt = record.get("report_text")
    if not isinstance(rt, dict):
        return str(rt or "")
    parts = []
    for key in ["ClinicalInformation_EN", "Technique_EN", "Findings_EN", "Impressions_EN"]:
        val = rt.get(key)
        if val:
            parts.append(f"[{key}]\n{val}")
    return "\n\n".join(parts)


def load_false_negatives():
    rows = []
    with open(FALSE_NEG_CSV) as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows.append(r)
    return rows


def load_existing_results():
    if not RESULTS_CSV.exists():
        return {}
    done = {}
    with open(RESULTS_CSV) as f:
        reader = csv.DictReader(f)
        for r in reader:
            done[(r["label"], r["item_id"])] = r["judgment"]
    return done


def append_result(label, item_id, predicted_prob, judgment, note):
    is_new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        writer = csv.writer(f)
        if is_new:
            writer.writerow(["label", "item_id", "predicted_prob", "judgment", "note", "reviewer"])
        writer.writerow([label, item_id, predicted_prob, judgment, note, REVIEWER])


manifest = load_manifest()
false_negatives = load_false_negatives()
available_labels = sorted({r["label"] for r in false_negatives})
print(f"可核對的類別（共 {len(available_labels)} 類，每類最多 5 張假陰性）：")
for name in available_labels:
    n = sum(1 for r in false_negatives if r["label"] == name)
    print(f"  - {name} ({n} 張)")

可核對的類別（共 17 類，每類最多 5 張假陰性）：
  - Arterial wall calcification (5 張)
  - Atelectasis (5 張)
  - Bronchiectasis (5 張)
  - Cardiomegaly (5 張)
  - Consolidation (5 張)
  - Coronary artery wall calcification (5 張)
  - Emphysema (5 張)
  - Interlobular septal thickening (5 張)
  - Lung nodule (5 張)
  - Lung opacity (5 張)
  - Lymphadenopathy (5 張)
  - Medical material (5 張)
  - Mosaic attenuation pattern (5 張)
  - Peribronchial thickening (5 張)
  - Pericardial effusion (5 張)
  - Pleural effusion (5 張)
  - Pulmonary fibrotic sequela (5 張)


In [ ]:
label_dropdown = widgets.Dropdown(options=available_labels, description="要核對的類別：", style={"description_width": "initial"})
display(label_dropdown)

Dropdown(description='要核對的類別：', options=('Arterial wall calcification', 'Atelectasis', 'Bronchiectasis', 'Card…

In [ ]:
# state
selected_label = label_dropdown.value
existing = load_existing_results()
queue = [r for r in false_negatives if r["label"] == selected_label]
state = {"idx": 0}

title_html = widgets.HTML()
image_widget = widgets.Image(format="png", width=420)
report_box = widgets.Textarea(layout=widgets.Layout(width="600px", height="260px"), disabled=True)
note_box = widgets.Text(placeholder="（選填）備註", layout=widgets.Layout(width="400px"))
progress_html = widgets.HTML()

btn_visible = widgets.Button(description="看得到病灶（模型問題）", button_style="danger", layout=widgets.Layout(width="220px"))
btn_not_visible = widgets.Button(description="看不到病灶（切片天花板）", button_style="success", layout=widgets.Layout(width="220px"))
btn_uncertain = widgets.Button(description="不確定", button_style="warning", layout=widgets.Layout(width="120px"))
btn_prev = widgets.Button(description="⬅ 上一筆", layout=widgets.Layout(width="100px"))
btn_skip = widgets.Button(description="跳過（不記錄）➡", layout=widgets.Layout(width="140px"))


def render():
    idx = state["idx"]
    if idx >= len(queue):
        title_html.value = "<h3>這個類別的假陰性樣本都看完了 🎉</h3>"
        image_widget.value = b""
        report_box.value = ""
        progress_html.value = f"{len(queue)}/{len(queue)}"
        return

    row = queue[idx]
    item_id = row["item_id"]
    record = manifest.get(item_id, {})
    png_path = REX_DIR / record.get("image_png", "")

    already = existing.get((selected_label, item_id))
    badge = f" &nbsp; <b style='color:gray'>[已判斷過：{already}]</b>" if already else ""

    title_html.value = (
        f"<h4>{selected_label} — {item_id}</h4>"
        f"<p>模型預測機率：<b>{row['predicted_prob']}</b>（真實標籤為陽性，機率越低代表模型越確信是陰性，"
        f"也就是漏判越嚴重）{badge}</p>"
    )

    if png_path.exists():
        image_widget.value = png_path.read_bytes()
    else:
        image_widget.value = b""
        title_html.value += f"<p style='color:red'>找不到圖檔：{png_path}</p>"

    report_box.value = full_report_text(record)
    note_box.value = ""
    progress_html.value = f"第 {idx + 1} / {len(queue)} 張"


def make_judge_handler(judgment):
    def handler(_btn):
        idx = state["idx"]
        if idx >= len(queue):
            return
        row = queue[idx]
        append_result(selected_label, row["item_id"], row["predicted_prob"], judgment, note_box.value)
        existing[(selected_label, row["item_id"])] = judgment
        state["idx"] += 1
        render()
    return handler


def on_prev(_btn):
    state["idx"] = max(0, state["idx"] - 1)
    render()


def on_skip(_btn):
    state["idx"] += 1
    render()


btn_visible.on_click(make_judge_handler("visible_model_issue"))
btn_not_visible.on_click(make_judge_handler("not_visible_ceiling"))
btn_uncertain.on_click(make_judge_handler("uncertain"))
btn_prev.on_click(on_prev)
btn_skip.on_click(on_skip)

render()

display(
    widgets.VBox([
        progress_html,
        title_html,
        widgets.HBox([image_widget, report_box]),
        note_box,
        widgets.HBox([btn_visible, btn_not_visible, btn_uncertain, btn_prev, btn_skip]),
    ])
)

## 查看目前為止的判斷結果統計

In [ ]:
import collections

if RESULTS_CSV.exists():
    with open(RESULTS_CSV) as f:
        rows = list(csv.DictReader(f))
    by_label = collections.defaultdict(collections.Counter)
    for r in rows:
        by_label[r["label"]][r["judgment"]] += 1
    for label, counter in by_label.items():
        print(label, dict(counter))
else:
    print("還沒有任何判斷結果")